[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/cnn.ipynb)

# Convolutional Neural Networks

Convolutional neural networks are very important in several areas, such as image processing. Their characteristics allow building networks with many layers (deep networks), maintaining a "reduced" number of parameters.

## Types of layers

There are three main types of layers: convolutional, pooling and fully connected. We'll look into an example, to assess the differences in terms of input and output dimensions.

For a convolutional (or pooling) layer, the spatial size of the output is given by

$$W_{out} = \left\lfloor \frac{W_{in} + 2P - K}{S} \right\rfloor + 1$$

where $K$ is the kernel size, $P$ the padding and $S$ the stride (the same formula applies to the height). The number of output channels is the number of filters (`out_channels`).

Importing libraries:

In [ ]:
# Importing libraries
import torch
import torch.nn as nn

Defining a single example: a random 3-channel (e.g., RGB) image of 39x39 pixels and its class label. PyTorch expects images in the format (N, C, H, W): batch size, channels, height and width.

In [ ]:
torch.manual_seed(20)

X_train = torch.randn(1, 3, 39, 39)  # one image, 3 channels, 39x39 pixels
y_train = torch.tensor([3])          # its class (one of 10 classes)
print(X_train.shape)
print(y_train.shape)

## Building the model

Here we'll build the model. We will not train the model... just checking the operations. Using the formula above:

* `cnn1`: $(39 + 0 - 2)/1 + 1 = 38$ → 10 x 38 x 38
* `cnn2`: $\lfloor(38 + 0 - 5)/2\rfloor + 1 = 17$ → 20 x 17 x 17
* `cnn3`: $\lfloor(17 + 0 - 5)/2\rfloor + 1 = 7$ → 40 x 7 x 7
* flatten: 40 x 7 x 7 = 1960 values, fed to a fully connected layer with 10 outputs

In [ ]:
# Creating the Model
class CNN_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn1 = nn.Conv2d(in_channels=3, out_channels=10, kernel_size=2, stride=1, padding=0)
        self.cnn2 = nn.Conv2d(in_channels=10, out_channels=20, kernel_size=5, stride=2, padding=0)
        self.cnn3 = nn.Conv2d(in_channels=20, out_channels=40, kernel_size=5, stride=2, padding=0)
        self.fc1 = nn.Linear(40*7*7,10)
        
    def forward(self,x):
        # Convolution 1
        print(x.shape)
        out = self.cnn1(x)
        print(out.shape)

        # Convolution 2
        out = self.cnn2(out)
        print(out.shape)


        # Convolution 3
        out = self.cnn3(out)
        print(out.shape)

        # Flatten all dimensions except the batch (same as out.view(out.size(0), -1))
        out = torch.flatten(out, 1)
        print(out.shape)

        out = self.fc1(out)
        print(out.shape)
        
        return out

In [ ]:
torch.manual_seed(20)
model = CNN_model()

In [ ]:
print(model)
for name, p in model.named_parameters():
    print(f"{name:12s} {str(tuple(p.shape)):18s} {p.numel():6d} parameters")
print("Total:", sum(p.numel() for p in model.parameters()))

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)

In [ ]:
y_pred = model(X_train)  # calling the model runs forward() (don't call model.forward() directly)
loss = loss_function(y_pred, y_train)
optimizer.zero_grad()
loss.backward()
optimizer.step()
print("Loss:", loss.item())

## Pooling layers

The model above reduces the spatial size using strides. The more common alternative is to use a pooling layer, which has no parameters: it just keeps the maximum (or average) of each window. With `kernel_size=2, stride=2` the height and width are halved, while the number of channels is kept.

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=10, kernel_size=3, padding=1)  # "same" padding: keeps 39x39
pool = nn.MaxPool2d(kernel_size=2, stride=2)

out = conv(X_train)
print("after conv:", out.shape)
out = pool(out)
print("after pool:", out.shape)  # floor((39 - 2)/2) + 1 = 19
print("pooling parameters:", sum(p.numel() for p in pool.parameters()))

## Why so few parameters?

Each filter is shared across all the positions of the image (weight sharing), so the number of parameters only depends on the kernel size and the number of channels, not on the image size. Let's compare the first convolutional layer with a fully connected layer producing an output of the same size.

In [ ]:
conv_params = sum(p.numel() for p in model.cnn1.parameters())

n_in = 3 * 39 * 39    # input values
n_out = 10 * 38 * 38  # output values of cnn1
dense_params = n_in * n_out + n_out  # weights + biases of an equivalent nn.Linear(n_in, n_out)

print(f"Conv2d(3, 10, kernel_size=2): {conv_params:,} parameters")
print(f"Linear({n_in}, {n_out}):     {dense_params:,} parameters")